# GroupProject — Fixed Colab runner (Smart Invigilation)

**Reg: GroupProject**

This version fixes the failures in the previous Colab copy:
1. Finds the **shared** SmartInvigilation Drive folder (or downloads it)
2. Falls back to **Kaggle** ExamCheatingDataset without hanging
3. Trains EfficientNet-B0/B3 and writes **`best.pt` / `GroupProject_best.pt`** into Drive

**Runtime → Change runtime type → GPU** (TPU also OK). Then **Runtime → Run all**.


In [ ]:
# 0) Install + imports
import os, sys, subprocess, re, io, json, math, time, random, copy
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "imagehash", "gdown", "kaggle", "opencv-python-headless"], check=False)

import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
import torchvision.transforms as T
from torchvision import models
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix, precision_recall_curve
import imagehash

AUTHOR = "GroupProject"
SHARED_FOLDER_ID = "1tcj53raozUaS2jx3-BpbKT5ITUB1G1F5"
print("OK | torch", torch.__version__, "| cuda", torch.cuda.is_available())


In [ ]:
# 1) Mount Drive + locate SmartInvigilation (shared folder safe)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

def find_smart_root():
    candidates = [
        Path("/content/drive/MyDrive/SmartInvigilation"),
        Path(f"/content/drive/MyDrive/.shortcut-targets-by-id/{SHARED_FOLDER_ID}"),
        Path(f"/content/drive/.shortcut-targets-by-id/{SHARED_FOLDER_ID}"),
        Path("/content/SmartInvigilation"),
    ]
    # Any shortcut named SmartInvigilation
    for p in Path("/content/drive/MyDrive").glob("*"):
        if p.is_dir() and "smartinvigilation" in p.name.lower():
            candidates.append(p)
    for c in candidates:
        if c.exists():
            return c
    # Search for known marker files/folders (bounded)
    drive_root = Path("/content/drive")
    markers = ["Exam_Cheating_TPU_Training.ipynb", "archive(1)", "examdatasetnew"]
    if drive_root.exists():
        for dirpath, dirnames, filenames in os.walk(drive_root):
            depth = dirpath[len(str(drive_root)):].count(os.sep)
            if depth > 5:
                dirnames[:] = []
                continue
            base = Path(dirpath)
            if any((base / m).exists() for m in markers) or "Exam_Cheating_TPU_Training.ipynb" in filenames:
                return base
    return None

DRIVE_ROOT = find_smart_root()
if DRIVE_ROOT is None:
    print("Shared folder not linked in MyDrive — downloading via gdown (may take a while)...")
    dest = Path("/content/SmartInvigilation")
    dest.mkdir(parents=True, exist_ok=True)
    import gdown
    gdown.download_folder(id=SHARED_FOLDER_ID, output=str(dest), quiet=False)
    DRIVE_ROOT = dest

print("DRIVE_ROOT =", DRIVE_ROOT)
print("top entries:", [p.name for p in Path(DRIVE_ROOT).iterdir()][:30])


In [ ]:
# 2) Config (auto-adjust paths that exist)
def first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return str(Path(p))
    return None

exam_paths = [
    Path(DRIVE_ROOT) / "examdatasetnew" / "datasetnew",
    Path(DRIVE_ROOT) / "examdatasetnew",
    Path(DRIVE_ROOT) / "datasetnew",
]
ecd_paths = [
    Path(DRIVE_ROOT) / "archive(1)" / "ExamCheatingDataset" / "train",
    Path(DRIVE_ROOT) / "ExamCheatingDataset" / "train",
    Path(DRIVE_ROOT) / "archive(1)" / "ExamCheatingDataset",
    Path(DRIVE_ROOT) / "ExamCheatingDataset",
]
exam_path = first_existing(exam_paths)
ecd_path = first_existing(ecd_paths)

OUT_DIR = Path("/content/drive/MyDrive/SmartInvigilation_runs/GroupProject_effnet")
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/content/work_GroupProject"); WORK_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_GPU = DEVICE.type == "cuda"

CFG = dict(
    AUTHOR=AUTHOR,
    DRIVE_ROOT=str(DRIVE_ROOT),
    OUT_DIR=str(OUT_DIR),
    WORK_DIR=str(WORK_DIR),
    NORMAL_CLASSES=["normal", "normal act", "not cheating", "notcheating"],
    FRONTWATCHING_SOFT=0.35,
    ARCH="efficientnet_b3" if USE_GPU else "efficientnet_b0",
    ARCH_FALLBACK="efficientnet_b0",
    IMG=288 if USE_GPU else 224,
    CACHE_SIZE=320,
    BATCH=32 if USE_GPU else 16,
    EPOCHS_HEAD=3, EPOCHS_FT=20,
    LR_HEAD=1e-3, LR_FT=8e-5, WD=1e-4,
    WARMUP_EPOCHS=1, GRAD_CLIP=1.0,
    PATIENCE=5, LOSS="weighted_ce",
    N_FOLDS=5, FOLDS_TO_RUN=3,   # 3 folds for Colab time; raise to 5 if you have time
    TARGET_PRECISION=0.80,
    EMA_DECAY=0.995, MIXUP_ALPHA=0.2, USE_TTA=True, EXPORT_ENSEMBLE=True,
    DROP_FLIP_COPY_FILES=True, DEDUP_HAMMING=4,
    YOLO_WEIGHTS="yolov8n.pt", YOLO_CONF=0.3,
    CROP_IF_PERSON_AREA_BELOW=0.6, CROP_PAD=0.12, MIN_CROP_PX=40,
    SEED=42,
)

SOURCES = []
if exam_path:
    SOURCES.append(dict(name="examdatasetnew", type="images", path=exam_path))
if ecd_path:
    SOURCES.append(dict(name="ExamCheatingDataset", type="images", path=ecd_path,
                        kaggle="ardutraagiginting/exam-cheating-dataset"))
else:
    SOURCES.append(dict(name="ExamCheatingDataset", type="images",
                        path=str(Path("/content/data/ExamCheatingDataset")),
                        kaggle="ardutraagiginting/exam-cheating-dataset"))

print("device:", DEVICE, "| arch:", CFG["ARCH"])
print("examdatasetnew:", exam_path)
print("ExamCheatingDataset:", ecd_path)
print("OUT_DIR:", OUT_DIR)
assert SOURCES, "No dataset sources found"



In [ ]:
# 3) Resolve sources (Kaggle fallback if needed)
def kaggle_download(slug, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    kj = Path.home() / ".kaggle/kaggle.json"
    if not kj.exists():
        # try Colab secrets / uploaded file locations
        for alt in [Path("/content/kaggle.json"), Path("/content/drive/MyDrive/kaggle.json")]:
            if alt.exists():
                kj.parent.mkdir(parents=True, exist_ok=True)
                kj.write_bytes(alt.read_bytes()); kj.chmod(0o600)
                break
    if not kj.exists():
        print("No kaggle.json — skipping Kaggle for", slug)
        return None
    subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", str(dest), "--unzip"], check=True)
    return dest

def src_root(s):
    p = Path(s["path"])
    if p.exists():
        # if this points at dataset root with train/, prefer train
        if (p / "train").is_dir():
            return p / "train"
        return p
    if s.get("kaggle"):
        dest = Path("/content/data") / s["name"]
        if not dest.exists():
            got = kaggle_download(s["kaggle"], dest)
            if got is None:
                return p
        # find class folders
        for cand in [dest, dest / "train", *dest.glob("**/train")]:
            if cand.is_dir() and any(cand.iterdir()):
                # class subdirs?
                sub = [x for x in cand.iterdir() if x.is_dir()]
                if sub:
                    return cand
        return dest
    return p

for s in SOURCES:
    r = src_root(s)
    print(s["name"], "->", r, "| exists:", r.exists())



In [ ]:
# 4) Build dataframe
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
norm = lambda s: re.sub(r"[_\-]+", " ", str(s).strip().lower())

def label_of(cls):
    c = norm(cls)
    if c in [norm(x) for x in CFG["NORMAL_CLASSES"]]:
        return 0.0
    if c == "frontwatching":
        return float(CFG["FRONTWATCHING_SOFT"])
    return 1.0

rows = []
for s in SOURCES:
    root = src_root(s)
    if not root.exists():
        print("MISSING:", s["name"], root); continue
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() not in IMG_EXT: continue
        rel = p.relative_to(root)
        if len(rel.parts) < 2: continue
        cls = norm(rel.parts[0])
        y = label_of(cls)
        rows.append(dict(path=str(p), dataset=s["name"], kind="images", cls=cls, y=y, y_hard=int(y >= 0.5),
                         subject=None, multi="largest", train=True))

df = pd.DataFrame(rows)
assert len(df) > 0, "No images found — add a Drive shortcut to the shared SmartInvigilation folder or provide kaggle.json"
print(df.groupby(["dataset", "cls"]).agg(n=("y", "size"), y_mean=("y", "mean")).to_string())

if CFG["DROP_FLIP_COPY_FILES"]:
    stem = df.path.map(lambda p: Path(p).stem.lower())
    bad = stem.str.contains(r"copy") | stem.str.contains(r"_f(?:\s*\(\d+\))?$")
    print("dropped flip/copy:", int(bad.sum()))
    df = df[~bad].reset_index(drop=True)



In [ ]:
# 5) Dedup + YOLO crops
def phashes(p):
    im = Image.open(p).convert("L")
    return int(str(imagehash.phash(im)), 16), int(str(imagehash.phash(ImageOps.mirror(im))), 16)

im_idx = np.arange(len(df))
H = np.array([phashes(df.path[i]) for i in im_idx], dtype=np.uint64).reshape(-1, 2)
h, hm = H[:, 0], H[:, 1]
popcnt = (lambda a: np.bitwise_count(a)) if hasattr(np, "bitwise_count") else (lambda a: np.unpackbits(a.view(np.uint8).reshape(*a.shape, 8), -1).sum(-1))
parent = list(range(len(im_idx)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
for i in range(len(im_idx)):
    d = np.minimum(popcnt(h[i] ^ h[i+1:]), popcnt(h[i] ^ hm[i+1:]))
    for j in np.nonzero(d <= CFG["DEDUP_HAMMING"])[0] + i + 1:
        parent[find(int(j))] = find(i)
df["subject"] = [f"img/g{find(k)}" for k in range(len(im_idx))]
print("rows", len(df), "groups", df.subject.nunique())

from ultralytics import YOLO
yolo = YOLO(CFG["YOLO_WEIGHTS"])
cache = Path(CFG["WORK_DIR"]) / f"crops{CFG['CACHE_SIZE']}"; cache.mkdir(parents=True, exist_ok=True)
S, out_rows, n_multi = CFG["CACHE_SIZE"], [], 0
for r in df.itertuples():
    im = Image.open(r.path).convert("RGB"); W, Hh = im.size
    res = yolo.predict(im, classes=[0], conf=CFG["YOLO_CONF"], device="cpu", verbose=False)[0]
    b = res.boxes.xyxy.cpu().numpy() if len(res.boxes) else np.zeros((0, 4))
    b = b[np.minimum(b[:, 2]-b[:, 0], b[:, 3]-b[:, 1]) >= CFG["MIN_CROP_PX"]]
    area = (b[:, 2]-b[:, 0]) * (b[:, 3]-b[:, 1])
    if len(b) > 1:
        n_multi += 1; b = b[[area.argmax()]]
    elif len(b) == 1 and area[0] / (W * Hh) >= CFG["CROP_IF_PERSON_AREA_BELOW"]:
        b = np.zeros((0, 4))
    crops = []
    for x1,y1,x2,y2 in b:
        px, py = CFG["CROP_PAD"]*(x2-x1), CFG["CROP_PAD"]*(y2-y1)
        crops.append(im.crop((max(0,x1-px), max(0,y1-py), min(W,x2+px), min(Hh,y2+py))))
    if not crops: crops = [im]
    for k,c in enumerate(crops):
        out = cache / f"{r.Index:06d}_{k}.jpg"
        c.resize((S,S), Image.BILINEAR).save(out, quality=95)
        out_rows.append({**r._asdict(), "crop": str(out)})
data = pd.DataFrame(out_rows).drop(columns=["Index"], errors="ignore")
print("crops:", len(data), "| multi:", n_multi)
print(data.groupby(["dataset","y_hard"]).size().unstack(fill_value=0))



In [ ]:
# 6) Folds, augs, model, train, export best.pt
CLASS_NAMES, CHEAT_CLASSES, CHEAT_IDX = ["normal", "cheating"], ["cheating"], [1]
data["t"] = data.y_hard
train_all = data[data.train].reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
train_all["fold"] = -1
_groups = train_all.subject.where(train_all.subject.notna(), train_all.index.map(lambda i: f"row{i}"))
for f, (_, va) in enumerate(sgkf.split(train_all, train_all.t, groups=_groups)):
    train_all.loc[va, "fold"] = f
print(pd.crosstab(train_all.fold, train_all.y_hard))

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
class DownUp:
    def __call__(self, im):
        if random.random() > 0.25: return im
        s = random.randint(max(16, CFG["IMG"]//4), max(17, CFG["IMG"]//2))
        return im.resize((s,s), Image.BILINEAR).resize(im.size, Image.BILINEAR)
class Jpeg:
    def __call__(self, im):
        if random.random() > 0.3: return im
        b = io.BytesIO(); im.save(b, "JPEG", quality=random.randint(28,92)); b.seek(0)
        return Image.open(b).convert("RGB")
train_tf = T.Compose([T.RandomResizedCrop(CFG["IMG"], scale=(0.55,1.0)), T.RandomHorizontalFlip(),
    T.RandomApply([T.RandomRotation(12)], p=0.5), T.ColorJitter(0.35,0.35,0.25,0.04),
    T.RandomApply([T.GaussianBlur(5,(0.1,2.0))], p=0.25), T.RandomAutocontrast(p=0.2),
    DownUp(), Jpeg(), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((CFG["IMG"], CFG["IMG"])), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf_flip = T.Compose([T.Resize((CFG["IMG"], CFG["IMG"])), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(MEAN, STD)])

def mixup_batch(x, y, alpha):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

class Crops(Dataset):
    def __init__(self, d, tf):
        self.p, self.t, self.tf = d.crop.tolist(), d.t.tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert("RGB")), self.t[i]

NW = 2
def train_loader(d):
    return DataLoader(Crops(d, train_tf), CFG["BATCH"], shuffle=True, drop_last=True, num_workers=NW)

@torch.no_grad()
def predict(model, d, tta=True):
    model.eval(); outs=[]
    def one(tf):
        o=[]
        for x,_ in DataLoader(Crops(d, tf), CFG["BATCH"], num_workers=NW):
            n=len(x)
            if n < CFG["BATCH"]: x=torch.cat([x, x.new_zeros(CFG["BATCH"]-n, *x.shape[1:])])
            p=torch.softmax(model(x.to(DEVICE)).float(),1)[:,CHEAT_IDX].sum(1)
            o.append(p.cpu()[:n])
        return torch.cat(o).numpy()
    p0 = one(eval_tf)
    return 0.5*(p0+one(eval_tf_flip)) if (tta and CFG["USE_TTA"]) else p0

def build_model(n, arch=None):
    arch = arch or CFG["ARCH"]
    try:
        if arch == "efficientnet_b3":
            m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, n); head=m.classifier
        else:
            m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, n); head=m.classifier
            CFG["ARCH"]="efficientnet_b0"; CFG["IMG"]=min(CFG["IMG"],224)
        return m, head, CFG["ARCH"]
    except Exception as e:
        print("fallback B0:", e); CFG["ARCH"]="efficientnet_b0"; return build_model(n, "efficientnet_b0")

class ModelEMA:
    def __init__(self, model, decay):
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)
        self.decay=decay
    @torch.no_grad()
    def update(self, model):
        d=self.decay; msd=model.state_dict()
        for k,v in self.ema.state_dict().items():
            if v.dtype.is_floating_point: v.mul_(d).add_(msd[k].detach(), alpha=1-d)
            else: v.copy_(msd[k])

def metrics(y,p,thr=0.5):
    yh=(p>=thr).astype(int)
    return dict(threshold=float(thr),
                precision=float(precision_score(y,yh,zero_division=0)),
                recall=float(recall_score(y,yh,zero_division=0)),
                macro_f1=float(f1_score(y,yh,average="macro",zero_division=0)),
                ap=float(average_precision_score(y,p)) if len(set(y))>1 else float("nan"),
                confusion=confusion_matrix(y,yh,labels=[0,1]).tolist(), n=int(len(y)))

def fit_temperature(y,p,grid=np.linspace(0.5,3.0,26)):
    p=np.clip(p,1e-6,1-1e-6); z=np.log(p/(1-p)); best_t,best_ap=1.0,-1
    for t in grid:
        pt=1/(1+np.exp(-z/t)); ap=average_precision_score(y,pt) if len(set(y))>1 else 0
        if ap>best_ap: best_ap,best_t=ap,float(t)
    return best_t

def apply_temperature(p,t):
    p=np.clip(p,1e-6,1-1e-6); z=np.log(p/(1-p)); return 1/(1+np.exp(-z/t))

def threshold_for_precision(y,p,target):
    pr,rc,th=precision_recall_curve(y,p); ok=np.nonzero(pr[:-1]>=target)[0]
    return float(th[ok[np.argmax(rc[ok])]]) if len(ok) else 0.5

def train(tr, va=None, epochs_ft=None):
    torch.manual_seed(CFG["SEED"]); random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
    model, head, used = build_model(len(CLASS_NAMES)); model.to(DEVICE); CFG["ARCH"]=used
    ema = ModelEMA(model, CFG["EMA_DECAY"])
    cnt=np.bincount(tr.t, minlength=2).astype(float)
    cw=torch.tensor(cnt.sum()/np.maximum(cnt,1)/len(cnt), dtype=torch.float32, device=DEVICE)
    loss_fn=nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)
    dl=train_loader(tr)
    best,best_state,best_ep,bad=-1,None,0,0
    stages=[("head",CFG["EPOCHS_HEAD"],CFG["LR_HEAD"]),("ft",epochs_ft or CFG["EPOCHS_FT"],CFG["LR_FT"])]
    for stage,n_ep,lr in stages:
        for p in model.parameters(): p.requires_grad=(stage=="ft")
        for p in head.parameters(): p.requires_grad=True
        opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=CFG["WD"])
        steps=max(n_ep*len(dl),1); warm=CFG["WARMUP_EPOCHS"]*len(dl) if stage=="ft" else 0
        def lr_at(step):
            if step<warm: return lr*(step+1)/max(warm,1)
            prog=(step-warm)/max(steps-warm,1); return lr*0.5*(1+math.cos(math.pi*prog))
        step_i=0
        for ep in range(n_ep):
            model.train(); tot=0.0; t0=time.time()
            for x,t in dl:
                x,t=x.to(DEVICE),t.to(DEVICE)
                for g in opt.param_groups: g["lr"]=lr_at(step_i)
                xm,ya,yb,lam=mixup_batch(x,t, CFG["MIXUP_ALPHA"] if stage=="ft" else 0.0)
                logits=model(xm).float()
                loss = lam*loss_fn(logits,ya)+(1-lam)*loss_fn(logits,yb) if stage=="ft" and CFG["MIXUP_ALPHA"]>0 else loss_fn(logits,t)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
                opt.step(); ema.update(model); step_i+=1; tot+=float(loss.detach().cpu())
            rec=dict(stage=stage, epoch=ep, loss=tot/max(len(dl),1), sec=round(time.time()-t0,1))
            if va is not None and stage=="ft":
                rec["val_macro_f1"]=metrics(va.y_hard.values, predict(ema.ema, va))["macro_f1"]
                if rec["val_macro_f1"]>best:
                    best,best_ep,bad=rec["val_macro_f1"],ep+1,0
                    best_state={k:v.detach().cpu().clone() for k,v in ema.ema.state_dict().items()}
                else: bad+=1
            print(rec)
            if va is not None and stage=="ft" and bad>=CFG["PATIENCE"]: break
    if best_state is not None:
        model.load_state_dict(best_state); ema.ema.load_state_dict(best_state)
    return ema.ema, best_ep or (epochs_ft or CFG["EPOCHS_FT"])

oof=np.full(len(train_all), np.nan); fold_res,best_eps,fold_states=[],[],[]
for f in range(min(CFG["FOLDS_TO_RUN"], CFG["N_FOLDS"])):
    tr,va=train_all[train_all.fold!=f], train_all[train_all.fold==f]
    print(f"--- fold {f}: train {len(tr)} val {len(va)}")
    model,ep=train(tr,va)
    oof[va.index]=predict(model,va); best_eps.append(ep)
    fold_states.append({k:v.detach().cpu().clone() for k,v in model.state_dict().items()})
    fold_res.append(dict(fold=f,best_epoch=ep, **metrics(va.y_hard.values, oof[va.index])))

m=~np.isnan(oof)
TEMP=fit_temperature(train_all.y_hard.values[m], oof[m])
oof_cal=apply_temperature(oof[m], TEMP)
THRESH=threshold_for_precision(train_all.y_hard.values[m], oof_cal, CFG["TARGET_PRECISION"])
cv=dict(temperature=TEMP, at_0_5=metrics(train_all.y_hard.values[m], oof[m]),
        at_calibrated=metrics(train_all.y_hard.values[m], oof_cal, THRESH))
print("OOF @0.5", cv["at_0_5"])
print("OOF calibrated", cv["at_calibrated"])

# ensemble export
out={k:torch.zeros_like(v,dtype=torch.float32) for k,v in fold_states[0].items() if v.dtype.is_floating_point}
other={k:v for k,v in fold_states[0].items() if not v.dtype.is_floating_point}
for st in fold_states:
    for k in out: out[k]+=st[k].float()
for k in out: out[k]/=len(fold_states)
out.update(other)
final,_,_=build_model(len(CLASS_NAMES)); final.load_state_dict(out, strict=False)

ckpt=dict(author=AUTHOR, arch=CFG["ARCH"], model_state_dict={k:v.detach().cpu() for k,v in final.state_dict().items()},
          class_names=CLASS_NAMES, cheating_classes=CHEAT_CLASSES, head="binary",
          input_size=CFG["IMG"], mean=MEAN, std=STD, threshold=THRESH, temperature=TEMP,
          export=f"fold_ema_ensemble_n={len(fold_states)}", cfg=CFG)
torch.save(ckpt, f"{CFG['OUT_DIR']}/best.pt")
torch.save(ckpt, f"{CFG['OUT_DIR']}/{AUTHOR}_best.pt")
# also drop copies into the shared SmartInvigilation folder if writable
for dest_dir in [Path(DRIVE_ROOT), Path(DRIVE_ROOT)/"models"]:
    try:
        dest_dir.mkdir(parents=True, exist_ok=True)
        torch.save(ckpt, dest_dir / "best.pt")
        torch.save(ckpt, dest_dir / f"{AUTHOR}_best.pt")
        print("also saved into", dest_dir)
    except Exception as e:
        print("could not write into", dest_dir, e)

json.dump(dict(author=AUTHOR, cv_folds=fold_res, cv=cv, threshold=THRESH, temperature=TEMP, config=CFG),
          open(f"{CFG['OUT_DIR']}/metrics_{AUTHOR}.json","w"), indent=2, default=str)
print("SAVED:", CFG["OUT_DIR"]+"/best.pt")
print("SAVED:", CFG["OUT_DIR"]+f"/{AUTHOR}_best.pt")
print("DONE — GroupProject")



## Done
Check Drive folder `MyDrive/SmartInvigilation_runs/GroupProject_effnet/` for:
- `best.pt`
- `GroupProject_best.pt`
- `metrics_GroupProject.json`

If the shared folder is writable, copies are also under that folder / `models/`.
